# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

os.environ["JAX_PLATFORM_NAME"] = "cpu"  # must be set before importing jax

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
from matplotlib.collections import LineCollection
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.ticker as mticker
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.skewers.skewers import SkewerBundle
from skewerjax.cosmology.cosmology import Cosmology

from skewerjax.fields.fields import ScalarField, VectorField, Fields
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux
import skewerjax.utils.plotting_tools as pt

import skewerjax.optical_depth.optical_depth_velocity_units as od

from skewerjax.constants.physical_constants import phys

# Cosmology

In [ ]:
# Create cosmology object
cosmology = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
# Define which cosmology to use, this is set as default to 0 since skewer and cosmology objects can deal with multiple cosmologies at once, 
# but other classes such as fields and optical depth are currently only set up for a single cosmology at a time
ii_cosmology = 0

# Lines

In [ ]:
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "HI-LyA-1206"]) # "SiIII-1206", "SiII-1190", "SiII-1193" # "HI-LyA-1206", "HI-LyA-1193", "HI-LyA-1190"

# Fields

In [ ]:
L_box = 110.7  # Mpc

In [ ]:
N_absorvers = 6
Delta = 0.5

n0_HI = 1e-11
T0_HI = 3e5

n0_SiIII = 1e-11
T0_SiIII = 3e5

n0_SiII = 1e-11
T0_SiII = 3e5

fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-5, C0=T0_HI, L_box=L_box, base_seed=0)),    # UNITS: [K]
        "number_density": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-20, C0=n0_HI, L_box=L_box, base_seed=0)), # [cm^-3]
        "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    },
    # "SiIII": {
    #     "temperature": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-5, C0=T0_SiIII, L_box=L_box, base_seed=0)),    # UNITS: [K]
    #     "number_density": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-20, C0=n0_SiIII, L_box=L_box, base_seed=0)), # [cm^-3]
    #     "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
    #     "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
    #     "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    # },
    # "SiII": {
    #     "temperature": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-5, C0=T0_SiII, L_box=L_box, base_seed=0)),    # UNITS: [K]
    #     "number_density": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-20, C0=n0_SiII, L_box=L_box, base_seed=0)), # [cm^-3]
    #     "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
    #     "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
    #     "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    # },
})

In [ ]:
# T0_HI = 1e5
# T0_Si = 2e6
# n0_min = 1e-16
# n0_max = 1e-10
# def pk_template(k: jnp.ndarray, A, n, k0, k_cut):
#     # P(k) = A (k/k0)^n exp[-(k/k_cut)^2]
#     kk = jnp.maximum(k, 1e-12)
#     return A * (kk / k0) ** n * jnp.exp(-(kk / k_cut) ** 2)
# number_density_sampler = toy_samplers.number_density_sampler_gaussian_3d(
#     key=jax.random.PRNGKey(0),
#     pk_of_k=lambda k: pk_template(k, A=5.0, n=-2.5, k0=2.0, k_cut=50.0),
#     n0_min=n0_min, n0_max=n0_max, L_box=L_box,
#     grid_shape=(256, 256, 256), contrast=1.2,
# )
# fields = Fields({
#     "HI": {
#         "temperature": ScalarField(toy_samplers.temperature_sampler(T0_HI, T0_HI, L_box)),    # UNITS: [K]
#         "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
#         "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
#         "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
#         "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
#     },
#     "SiIII": {
#         "temperature": ScalarField(toy_samplers.temperature_sampler(T0_Si, T0_Si, L_box)),    # UNITS: [K]
#         "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
#         "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
#         "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
#         "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
#     },
#     # "SiII": {
#     #     "temperature": ScalarField(toy_samplers.temperature_sampler(T0_Si, T0_Si, L_box)),    # UNITS: [K]
#     #     "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
#     #     "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
#     #     "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
#     #     "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
#     # },
# })

# Skewers

In [ ]:
z_box = 2.0

scale_factor_right  = 1 / (1 + z_box)
scale_factor_left = float(cosmology.solve_a_of_x(ICs=jnp.array([1/(1+z_box)]), x_eval_Mpc=jnp.linspace(start=L_box, stop=0.0, num=2))[0, -1])

In [ ]:
N_grid_skewers = 13
N_ppsk = 1024
points_grid_skewers = aux.make_skewer_grid_points(L_box, N_grid_skewers, N_ppsk, direction="y")
points = jnp.reshape(points_grid_skewers, (N_grid_skewers**2, N_ppsk, 3))

xx_LoS = jnp.linspace(start=0.0, stop=L_box, num=N_ppsk)
scale_factors = cosmology.solve_a_of_x(ICs=jnp.array([scale_factor_left]), x_eval_Mpc=xx_LoS)[0]

a0_per_skewer = scale_factors[N_ppsk-1]
x0_per_skewer = points_grid_skewers[0, 0, -1, 1]
skewer_bundle = SkewerBundle(spatial_points=points, cosmologies=cosmology, a0=a0_per_skewer, x0=x0_per_skewer)

In [ ]:
sampled_fields = fields.sample(skewer_bundle.quadri_vectors[ii_cosmology], skewer_bundle.los_directions)

# Optical depth

In [ ]:
a_dagger = 1 /  (1 + z_box)
a_em = a_dagger/skewer_bundle.scale_factors[0, 0, 0]
v_max = phys.c_kms * jnp.log(a_em)
velocities_kms = jnp.linspace(0.0, v_max, 1024)
optical_depth = od.compute_optical_depth_velocity_units(
    skewer_bundle=skewer_bundle, velocities_kms=velocities_kms, line_bundle=line_bundle, fields=fields, a_dagger=a_dagger,
)

# Plots

In [ ]:
%autoreload

In [ ]:
fig_nd, ax_nd = pt.plot_3d_radar_cylinder(
    points=points, sampled_fields=sampled_fields, field="number_density", mode="sampled_fields", skewer_index=0, n_samples_plot=1,
    cylinder_radius=1.5, cylinder_separation=5.2, field_logscale=True, normalization="global", scale_box_x_factor=0.83, legend_y_factor=0.8,
    scale_units_xy=(0.50, 1.05), profile_angle_offset_deg=90.0, n_y_slices=3, field_units=r"$\log_{10}\left(n \, \left[\mathrm{cm}^{-3}\right] \right)$",
    view_elev=18, view_azim=-40,
)
fig_cube, ax_cube = pt.plot_3d_neon_ridge_cube(
    points=points, sampled_fields=sampled_fields, field="number_density", mode="sampled_fields", normalization="global",
    field_logscale=True, profile_angle_offset_deg=-90.0, ridge_scale=1.2, elev=13, azim=-70
)
fig_tau, ax_tau = pt.plot_3d_radar_cylinder(
    points=points, mode="optical_depth", optical_depth=optical_depth, line_bundle=line_bundle, skewer_index=0, n_samples_plot=1, cylinder_radius=1.5,
    cylinder_separation=5.2, field_logscale=True, normalization="global", scale_box_x_factor=0.83, legend_y_factor=0.8, scale_units_xy=(0.50, 1.05),
    profile_angle_offset_deg=-90.0, n_y_slices=3, field_units=r"$\log_{10} \, \tau$", view_elev=18, view_azim=-40,
)
fig_tau_cube, ax_tau_cube = pt.plot_3d_neon_ridge_cube(
    points=points, mode="optical_depth", optical_depth=optical_depth, line_bundle=line_bundle, skewer_index=0, normalization="skewer",
    field_logscale=True, profile_angle_offset_deg=90.0, ridge_scale=1.2, elev=13, azim=-70,
)
fig, axes, idx = pt.plot_transmission_samples_neon(
    F_over_F0=np.exp(-optical_depth), line_bundle=line_bundle, x_axis_array=velocities_kms, x_label=r"$v \, \left[\mathrm{km} \, \mathrm{s}^{-1}\right]$", x_axis_key="velocities_kms",
    scale_factor_left=scale_factor_left, scale_factor_right=scale_factor_right, lambda0_anchor=None, N_plot_sample=4, seed=0
)

In [ ]:
_tau_factor = 1e0
array_tau_factors = np.array([1.0, _tau_factor], dtype=object) # One entry per line in tau_loaded. Use None for lines you want to discard

keep_mask = np.array([f is not None for f in array_tau_factors], dtype=bool) # Keep only entries with a valid scaling factor
tau_factors_kept = np.array(array_tau_factors[keep_mask], dtype=float) # Convert kept factors to float
tau_scaled = optical_depth[keep_mask] * tau_factors_kept[:, None, None] # Filter tau_loaded and apply the corresponding scaling

In [ ]:
import numpy as np
from itertools import combinations
from collections.abc import Mapping
from typing import TYPE_CHECKING, Dict, List, Sequence, Union

if TYPE_CHECKING:
    from .lines import LineBundle


class OpticalDepthCube:
    """
    Container for an optical-depth array with an associated LineBundle.

    The stored optical-depth field is assumed to have shape:
        (N_lines, N_skewers, N_spectral)

    where the first axis follows exactly the order of `line_bundle.lines`.
    """

    def __init__(self, tau: np.ndarray, line_bundle: "LineBundle"):
        tau = np.asarray(tau, dtype=float)

        if tau.ndim != 3:
            raise ValueError("`tau` must have shape (N_lines, N_skewers, N_spectral).")
        if tau.shape[0] != line_bundle.n_lines:
            raise ValueError(
                f"Mismatch between tau.shape[0]={tau.shape[0]} and "
                f"line_bundle.n_lines={line_bundle.n_lines}."
            )

        self.tau = tau
        self.line_bundle = line_bundle

    # ────────── Basic metadata ──────────
    @property
    def shape(self) -> tuple:
        return self.tau.shape

    @property
    def n_lines(self) -> int:
        return self.tau.shape[0]

    @property
    def n_skewers(self) -> int:
        return self.tau.shape[1]

    @property
    def n_spectral(self) -> int:
        return self.tau.shape[2]

    @property
    def line_ids(self) -> List[str]:
        return [line.id for line in self.line_bundle.lines]

    @property
    def species_keys(self) -> List[str]:
        return self.line_bundle.species_keys

    @property
    def ref_line_id(self) -> str:
        return self.line_bundle.ref_line_id

    @property
    def delta_v(self) -> np.ndarray:
        """
        Reference-line velocity offsets for the bundle lines.

        Defined in the LineBundle as:
            delta_v_j = c * ln(lambda_ref / lambda_j)
        """
        return np.asarray(self.line_bundle.delta_v, dtype=float)

    @property
    def delta_v_dict(self) -> Dict[str, float]:
        """Dictionary version of `delta_v`, keyed by line ID."""
        return dict(zip(self.line_ids, self.delta_v))

    # ────────── Forwarded line-bundle helpers ──────────
    def lines_for_species(self, species: str):
        return self.line_bundle.lines_for_species(species)

    def indices_for_species(self, species: str) -> List[int]:
        return self.line_bundle.indices_for_species(species)

    # ────────── Internal utilities ──────────
    @staticmethod
    def _periodic_shift_1d(y, delta_x, x, period):
        """
        Periodically shift a 1D field y(x) using the convention

            y_shifted(x) = y(x - delta_x)
        """
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)

        if x.ndim != 1 or y.ndim != 1 or x.size != y.size:
            raise ValueError("`x` and `y` must be 1D arrays of the same length.")

        idx = np.argsort(x)
        x = x[idx]
        y = y[idx]

        x0 = x[0]
        x_query = ((x - delta_x - x0) % period) + x0

        return np.interp(x_query, x, y, period=period)

    def _resolve_per_line_values(self, values, name="values") -> np.ndarray:
        """
        Convert a per-line input into a 1D array of length n_lines.

        Accepted inputs
        ---------------
        - None: defaults to `self.delta_v`
        - scalar: broadcast to all lines
        - array-like of shape (n_lines,)
        - dict-like keyed by `self.line_ids`
        """
        if values is None:
            return np.asarray(self.delta_v, dtype=float)

        if np.isscalar(values):
            return np.full(self.n_lines, float(values), dtype=float)

        if isinstance(values, Mapping):
            missing = [k for k in self.line_ids if k not in values]
            if missing:
                raise KeyError(f"Missing keys in `{name}`: {missing}")
            return np.asarray([values[k] for k in self.line_ids], dtype=float)

        arr = np.asarray(values, dtype=float)
        if arr.shape != (self.n_lines,):
            raise ValueError(
                f"`{name}` must have shape ({self.n_lines},), got {arr.shape}."
            )
        return arr

    # ────────── Main transformations ──────────
    def periodic_shift(self, delta_x=None, x=None, period=None) -> np.ndarray:
        """
        Periodically shift the line-stacked optical-depth array.

        Parameters
        ----------
        delta_x : float, array-like, dict, or None
            Shift applied independently to each line.
            Accepted forms:
              - None: defaults to `self.delta_v`
              - scalar: same shift for every line
              - array of shape (N_lines,)
              - dict keyed by bundle line IDs
        x : array, shape (N_spectral,)
            Spectral grid.
        period : float
            Period of the grid.

        Returns
        -------
        tau_shifted : ndarray, shape (N_lines, N_skewers, N_spectral)
        """
        if x is None:
            raise ValueError("`x` is required.")
        if period is None:
            raise ValueError("`period` is required.")

        x = np.asarray(x, dtype=float)
        if x.ndim != 1:
            raise ValueError("`x` must be 1D.")
        if x.size != self.n_spectral:
            raise ValueError(
                f"len(x)={x.size} must match the spectral axis size {self.n_spectral}."
            )

        delta_x = self._resolve_per_line_values(delta_x, name="delta_x")

        tau_flat = self.tau.reshape(self.n_lines, -1, self.n_spectral)
        shifted_flat = np.empty_like(tau_flat, dtype=float)

        for i_line in range(self.n_lines):
            dx = delta_x[i_line]
            for i_row in range(tau_flat.shape[1]):
                shifted_flat[i_line, i_row] = self._periodic_shift_1d(
                    y=tau_flat[i_line, i_row],
                    delta_x=dx,
                    x=x,
                    period=period,
                )

        return shifted_flat.reshape(self.tau.shape)

    def build_overflux(self, tau=None, ens_axis=0):
        """
        Build per-line overflux fields and the total overflux field.

        Definitions
        -----------
        For each line i:
            F_i = exp(-tau_i)
            delta_i = F_i / <F_i> - 1

        For the total field:
            F_tot = prod_i F_i
            delta_tot = F_tot / <F_tot> - 1

        Parameters
        ----------
        tau : array, shape (N_lines, N_skewers, N_spectral), optional
            Optical-depth array to process.
            If None, uses `self.tau`.
        ens_axis : int or tuple of int, default=0
            Axis (or axes) over which to average the total transmitted flux.
            Since F_tot has shape (N_skewers, N_spectral), using ens_axis=0
            reproduces your previous convention:
                Fbar_tot = mean(F_tot, axis=0)

        Returns
        -------
        overflux : ndarray, shape (N_lines, N_skewers, N_spectral)
            Per-line overflux fields.
        overflux_total : ndarray
            Total overflux field.
        """
        if tau is None:
            tau = self.tau

        tau = np.asarray(tau, dtype=float)

        if tau.ndim != 3:
            raise ValueError("`tau` must have shape (N_lines, N_skewers, N_spectral).")
        if tau.shape[0] != self.n_lines:
            raise ValueError(
                f"`tau.shape[0]` must match `self.n_lines={self.n_lines}`, "
                f"got {tau.shape[0]}."
            )

        F = np.exp(-tau)

        # Preserves your previous convention exactly
        Fbar = np.mean(F, axis=(1, 2), keepdims=True)
        overflux = F / Fbar - 1.0

        F_tot = np.prod(F, axis=0)

        if ens_axis is None:
            Fbar_tot = np.mean(F_tot)
        else:
            Fbar_tot = np.mean(F_tot, axis=ens_axis, keepdims=True)

        overflux_total = F_tot / Fbar_tot - 1.0

        return overflux, overflux_total

    def build_subset_fields(self, overflux, max_order=None, labels=None):
        """
        Build product fields for all non-empty subsets of the line axis.

        Parameters
        ----------
        overflux : array, shape (N_lines, N_skewers, N_spectral)
            Per-line overflux fields.
        max_order : int, optional
            Maximum subset size to include. If None, include all orders.
        labels : sequence, optional
            Labels associated with the line axis.
            If None, defaults to the bundle line IDs.

        Returns
        -------
        subset_fields : ndarray, shape (N_subsets, N_skewers, N_spectral)
            Product field for each subset.
        subset_labels : list[tuple]
            Labels describing each subset, in the same order as subset_fields.
        """
        overflux = np.asarray(overflux, dtype=float)

        if overflux.ndim != 3:
            raise ValueError("`overflux` must have shape (N_lines, N_skewers, N_spectral).")
        if overflux.shape[0] != self.n_lines:
            raise ValueError(
                f"`overflux.shape[0]` must match `self.n_lines={self.n_lines}`, "
                f"got {overflux.shape[0]}."
            )

        if max_order is None:
            max_order = self.n_lines
        if not (1 <= max_order <= self.n_lines):
            raise ValueError(
                f"`max_order` must satisfy 1 <= max_order <= {self.n_lines}."
            )

        if labels is None:
            labels = self.line_ids
        else:
            labels = list(labels)
            if len(labels) != self.n_lines:
                raise ValueError(
                    f"`labels` must have length {self.n_lines}, got {len(labels)}."
                )

        subset_arrays = []
        subset_labels = []

        for r in range(1, max_order + 1):
            for subset in combinations(range(self.n_lines), r):
                field = np.prod(overflux[list(subset)], axis=0)
                subset_arrays.append(field)
                subset_labels.append(tuple(labels[i] for i in subset))

        subset_fields = np.stack(subset_arrays, axis=0)
        return subset_fields, subset_labels

In [ ]:
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA") # "SiII-1190", "SiII-1193"
tau_cube = OpticalDepthCube(tau=tau_scaled, line_bundle=line_bundle)
tau_shifted = tau_cube.periodic_shift(x=velocities_kms, period=np.max(velocities_kms))

# list_key_names = [r"\alpha", r"X"]
list_key_names = ["1", "2"]
fields = {}
for ii in range(tau_shifted.shape[0]):
    fields[list_key_names[ii]] = np.exp(-tau_shifted[ii])

In [ ]:
overflux = sjx.skewers.skewers.DeltaFieldBuilder(fields).build()

In [ ]:
# ================================================================
# Plot fields 

rng = np.random.default_rng(42)
idx = rng.choice(overflux.field_total.shape[0], size=3, replace=False)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
linestyles = ["--", ":", "-.", (0, (3, 1, 1, 1)), (0, (5, 1))]

fig, axes = plt.subplots(figsize=(14, 6), ncols=1, nrows=2, sharex=True)
for j, i in enumerate(idx):
    c = colors[j % len(colors)]
    axes[0].plot(velocities_kms, overflux.field_total[i], lw=1.4, alpha=0.85, color=c, ls="-")
    axes[1].plot(velocities_kms, overflux.total[i], lw=1.4, alpha=0.85, color=c, ls="-")
    for m, label in enumerate(list(fields.keys())):
        ls = linestyles[m % len(linestyles)]
        axes[0].plot(velocities_kms, overflux.fields[label][i], lw=2.5, alpha=0.65, color=c, ls=ls)
        axes[1].plot(velocities_kms, overflux.delta_terms[label][i], lw=2.5, alpha=0.65, color=c, ls=ls)

axes[0].set_ylabel(r"$F$", fontsize=22)
axes[0].tick_params(axis="both", labelsize=22)
axes[1].set_xlabel(r"$v\, \left[\mathrm{km}/\mathrm{s}\right]$", fontsize=22)
axes[1].set_ylabel(r"$\delta$", fontsize=22)
axes[1].tick_params(axis="both", labelsize=22)

style_legend = [Line2D([0], [0], color="k", lw=2.0, ls="-", label=r"$F=\prod_j F_j$")]
for m, label in enumerate(list(fields.keys())):
    ls = linestyles[m % len(linestyles)]
    style_legend.append(Line2D([0], [0], color="k", lw=2.0, ls=ls, label=rf"$F_{{{label}}}$"))
axes[0].legend(handles=style_legend, fontsize=18, loc="lower left")

sample_legend = [Line2D([0], [0], color=colors[j % len(colors)], lw=3.0, ls="-", label=fr"sample #${i}$") for j, i in enumerate(idx)]
leg2 = axes[1].legend(handles=sample_legend, fontsize=18, loc="lower left")
axes[1].add_artist(leg2)

plt.tight_layout()
plt.show()

In [ ]:
Npix = tau_cube.shape[-1]

delta_v_list = list(np.array(line_bundle.delta_v))
L = np.max(velocities_kms) - np.min(velocities_kms)

In [ ]:
dv = np.diff(velocities_kms)[0]
stats = sjx.summary_stats.P1D_xi1D.OverfluxStatistics1D(overflux, dv)
ordered_pairs = stats.ordered_pairs

In [ ]:
stats.compute_p1d()

Pk_total_from_terms = stats.Pk_total_from_terms

Pk_total_per_sample = stats.Pk_total_per_sample
Pk_total = stats.Pk_total

Pk_pair_per_sample = stats.Pk_pair_per_sample
Pk_pair = stats.Pk_pair

k = stats.k

In [ ]:
# ================================================================
# Plot P1D

fig, axes, axes_res, order_sum_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_p1d_by_order(
    ordered_pairs=ordered_pairs, Pk_pair=Pk_pair, k=k, delta_v_list=delta_v_list, L_box=L, Npix=Npix,
    tolerance_threshold=1e-6,
    use_log_x=True,
    # xlim=(2.0, 600),
    residual_mode="sum",
    show_harmonics=True,
    use_symlog_main=True,
    use_symlog_residual=False,
    symlog_linscale=1.0,
    symlog_label_minor=False,
    symlog_sci_cutoff=1e-2,
    legend_ncol_terms=2,
    xlabel=r"$k\, L_\mathrm{box} / (2\pi)$", ylabel=r"$P_{\rm 1D}$", residual_ylabel=r"$\Delta$",
)

fig, (ax, ax_res), order_sum_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_p1d_total_recovery(
    Pk_total=Pk_total, Pk_total_from_terms=Pk_total_from_terms, Pk_pair=Pk_pair, k=k, delta_v_list=delta_v_list, L_box=L, Npix=Npix,
    ordered_pairs=ordered_pairs,
    show_harmonics=False,
    show_order_sums=True,
    show_order_sum_residuals=True,
    use_symlog_main=True,
    use_symlog_residual=False,
    xlabel=r"$k\, L_\mathrm{box} / (2\pi)$"
)

In [ ]:
# ===============================================================
# Build order sums from Pk_pair

ordered_pairs = list(Pk_pair.keys()) if ordered_pairs is None else ordered_pairs
order_sum_dict = {}
for pair in ordered_pairs:
    A, B = pair
    order = len(A) + len(B)
    if order not in order_sum_dict:
        order_sum_dict[order] = np.zeros_like(Pk_pair[pair], dtype=float)
    order_sum_dict[order] += Pk_pair[pair].real
orders = sorted(order_sum_dict.keys())

# ===============================================================
# Plot P1D

cmap = plt.get_cmap("tab10")

fig, ax = plt.subplots(figsize=(12, 9))
for i, order in enumerate(orders):
    y_order = order_sum_dict[order][1:] * k[1:] / np.pi
    ax.plot(k[1:], y_order, lw=2.0, color=cmap(i % 10), alpha=0.9, label=rf"$\mathcal{{O}}({order})$")

# Total
ax.plot(k[1:], Pk_total.real[1:] * k[1:] / np.pi, lw=3.0, color="k", label=r"Tot.")

# Total recovered from terms
ax.plot(k[1:], Pk_total_from_terms.real[1:] * k[1:] / np.pi, lw=2.0, color="red", ls="--", label=r"Tot. Sum")

ax.set_yscale("log")

ax.set_xlabel(r"$k$", fontsize=22)
ax.set_ylabel(r"$k P_{\rm 1D}(k) / \pi$", fontsize=22)

ax.set_xlim([0.0, 0.045])
ax.set_ylim([0.007, 0.6])

ax.tick_params(axis="both", labelsize=20)
ax.legend(fontsize=20, ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
stats.compute_xi1d()

xi_total_from_terms = stats.xi_total_from_terms

xi_total_per_sample = stats.xi_total_per_sample
xi_total = stats.xi_total

xi_pair_per_sample = stats.xi_pair_per_sample
xi_pair = stats.xi_pair

lags = stats.lags
lags_norm = lags / L

In [ ]:
# ================================================================
# Plot xi1D

fig, axes, axes_res, xi_order_sum_dict, xi_order_sum_plot_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_xi_by_order(
    ordered_pairs=ordered_pairs, xi_pair=xi_pair, lags_norm=lags_norm, delta_v_list=delta_v_list, L_box=L,
    tolerance_threshold=1e-6,
    residual_mode="sum",
    xlim=(-0.0, 0.5),
    show_delta_v_diff_lines=True,
    use_symlog_main=False,
    use_symlog_residual=False,
    symlog_linscale=1.0,
    symlog_base=10,
    symlog_label_minor=False,
    symlog_sci_cutoff=1e-2,
    show_order_badge=True,
    legend_ncol_terms=2,
    xlabel=r"$\Delta v / L_\mathrm{box}$", ylabel=r"$\xi_{\rm 1D}$", residual_ylabel=r"$\Delta$",
)

fig, (ax, ax_res), xi_order_sum_dict, xi_order_sum_plot_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_xi_total_recovery(
    xi_total=xi_total, xi_total_from_terms=xi_total_from_terms, xi_pair=xi_pair, lags_norm=lags_norm, delta_v_list=delta_v_list, L_box=L, ordered_pairs=ordered_pairs,
    xlim=(-0.0, 0.5),
    show_order_sums=True,
    show_order_sum_residuals=True,
    show_delta_v_diff_lines=True,
    annotate_half_line=False,
    half_line_text=r"$L_\mathrm{box}/2$",
    use_symlog_main=False,
    use_symlog_residual=True,
    main_symlog_linthresh=None,
    symlog_linscale=1.0,
    symlog_base=10,
    symlog_sci_cutoff=1e-2,
    xlabel=r"$\Delta v / L_\mathrm{box}$", ylabel=r"$\xi_{\rm 1D}$", residual_ylabel=r"$\Delta$",
)

# Periodic boundary conditions

In [ ]:
v_anchor = phys.c*1e-3 * np.log(scale_factor_obs / scale_factor_right)
delta_v_box = phys.c*1e-3 * jnp.log(scale_factor_right/scale_factor_left)
x_axis_array_anchor = x_axis_array - v_anchor

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 4))
colors = plt.cm.tab10(np.linspace(0, 1, len(line_bundle.lines)))
_idx = 0
for i_line, line in enumerate(line_bundle.lines):
    ax.plot(x_axis_array_anchor, F_over_F0[i_line, _idx, :], color=colors[i_line], lw=1.5, linestyle='-', alpha=1.0)
    x_left = phys.c*1e-3 * jnp.log(line.lambda0_A / lambda0_anchor)
    x_right = x_left + delta_v_box
    ax.axvline(x_left, color=colors[i_line], linestyle='--', lw=2.0, alpha=1.0)
    ax.axvline(x_right, color=colors[i_line], linestyle='--', lw=2.0, alpha=1.0)
ax.set_xlim(-2700, delta_v_box+500)
fig.tight_layout()
plt.show()

In [ ]:
def wrap_flux_lines_all_skewers(
    x_axis_array_anchor,
    F_over_F0,
    line_bundle,
    lambda0_anchor,
    delta_v_box,
):
    """
    Periodic wrapping of flux lines for all skewers.

    Parameters
    ----------
    x_axis_array_anchor : array, shape (n_pix,)
    F_over_F0 : array, shape (n_lines, n_skewers, n_pix)

    Returns
    -------
    x_axis_array_wrapped : dict[line_key] -> array, shape (n_keep,)
    F_over_F0_wrapped    : dict[line_key] -> array, shape (n_skewers, n_keep)
    line_intervals       : dict[line_key] -> metadata
    """
    x_axis_array_anchor = jnp.asarray(x_axis_array_anchor)
    F_over_F0 = jnp.asarray(F_over_F0)

    x_axis_array_wrapped = {}
    F_over_F0_wrapped = {}
    line_intervals = {}

    for i_line, line in enumerate(line_bundle.lines):
        line_key = str(line.name)

        x_left = phys.c * 1e-3 * jnp.log(line.lambda0_A / lambda0_anchor)
        x_right = x_left + delta_v_box

        # Same physical selection as in your original function
        mask = (x_axis_array_anchor >= x_left) & (x_axis_array_anchor < x_right)
        idx = jnp.where(mask)[0]

        x_crop = x_axis_array_anchor[idx]                         # (n_keep,)
        flux_crop = jnp.take(F_over_F0[i_line], idx, axis=-1)     # (n_skewers, n_keep)

        x_wrap = jnp.mod(x_crop, delta_v_box)
        order = jnp.argsort(x_wrap)

        x_axis_array_wrapped[line_key] = x_wrap[order]                  # (n_keep,)
        F_over_F0_wrapped[line_key] = jnp.take(flux_crop, order, axis=-1)  # (n_skewers, n_keep)

        line_intervals[line_key] = {
            "x_left": x_left,
            "x_right": x_right,
            "x_left_wrapped": jnp.mod(x_left, delta_v_box),
            "n_keep": len(idx),
        }

    return x_axis_array_wrapped, F_over_F0_wrapped, line_intervals

In [ ]:
x_axis_array_wrapped, F_over_F0_wrapped, line_intervals = wrap_flux_lines_all_skewers(
    x_axis_array_anchor=x_axis_array_anchor, F_over_F0=F_over_F0,
    line_bundle=line_bundle, lambda0_anchor=lambda0_anchor, delta_v_box=delta_v_box,
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 4))
colors = plt.cm.tab10(np.linspace(0, 1, len(line_bundle.lines)))
_idx = 0
for i_line, line in enumerate(line_bundle.lines):
    ax.plot(x_axis_array_wrapped[line.name], F_over_F0_wrapped[line.name][_idx, :], color=colors[i_line], lw=1.5, linestyle='-', alpha=1.0)
ax.set_xlim(-2700, delta_v_box+500)
fig.tight_layout()
plt.show()

In [ ]:
F_over_F0_wrapped.keys()

In [ ]:
F_over_F0_wrapped['Lyman-α'].shape

# Overflux

In [ ]:
fields = {}
labels = []
for ii, key in enumerate(F_over_F0_wrapped.keys()):
    _label = str(ii + 1)
    labels.append(_label)
    fields[_label] = F_over_F0_wrapped[key]

# ================================================================
# Overflux

overflux = sjx.skewers.skewers.DeltaFieldBuilder(fields).build()

In [ ]:
# ================================================================
# Plot fields 
rng = np.random.default_rng(0)
idx = rng.choice(overflux.field_total.shape[0], size=3, replace=False)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
linestyles = ["--", ":", "-.", (0, (3, 1, 1, 1)), (0, (5, 1))]

fig, axes = plt.subplots(figsize=(14, 6), ncols=1, nrows=2, sharex=True)
for j, i in enumerate(idx):
    c = colors[j % len(colors)]
    axes[0].plot(v, overflux.field_total[i], lw=1.4, alpha=0.85, color=c, ls="-")
    axes[1].plot(v, overflux.total[i], lw=1.4, alpha=0.85, color=c, ls="-")
    for m, label in enumerate(labels):
        ls = linestyles[m % len(linestyles)]
        axes[0].plot(v, fields[label][i], lw=2.5, alpha=0.65, color=c, ls=ls)
        axes[1].plot(v, overflux.delta_terms[label][i], lw=2.5, alpha=0.65, color=c, ls=ls)

axes[0].set_ylabel(r"$F$", fontsize=22)
axes[0].tick_params(axis="both", labelsize=22)
axes[1].set_xlabel(r"$v / L_\mathrm{box}$", fontsize=22)
axes[1].set_ylabel(r"$\delta$", fontsize=22)
axes[1].tick_params(axis="both", labelsize=22)

style_legend = [Line2D([0], [0], color="k", lw=2.0, ls="-", label=rf"$F=\prod_j\, F_j$")]
for m, label in enumerate(labels):
    ls = linestyles[m % len(linestyles)]
    style_legend.append(Line2D([0], [0], color="k", lw=2.0, ls=ls, label=rf"$F_{label}$"))
axes[0].legend(handles=style_legend, fontsize=18, loc="lower left")

sample_legend = [Line2D([0], [0], color=colors[j % len(colors)], lw=3.0, ls="-", label=fr"sample #${i}$") for j, i in enumerate(idx)]
leg2 = axes[1].legend(handles=sample_legend, fontsize=18, loc="lower left")
axes[1].add_artist(leg2)

plt.show()

In [ ]:

# ===============================================================
# Define 1D summary stats object

stats = sjx.summary_stats.P1D_xi1D.OverfluxStatistics1D(overflux, dv)
ordered_pairs = stats.ordered_pairs

# ===============================================================
# Compute P1D

stats.compute_p1d()

Pk_total_from_terms = stats.Pk_total_from_terms

Pk_total_per_sample = stats.Pk_total_per_sample
Pk_total = stats.Pk_total

Pk_pair_per_sample = stats.Pk_pair_per_sample
Pk_pair = stats.Pk_pair

k = stats.k

# ===============================================================
# Plots P1D

fig, axes, axes_res, order_sum_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_p1d_by_order(
    ordered_pairs=ordered_pairs, Pk_pair=Pk_pair, k=k, delta_v_list=delta_v_list, L_box=L_box, Npix=Npix,
    tolerance_threshold=1e-3,
    residual_mode="auto_reference",
    show_harmonics=True,
    use_symlog_main=True,
    use_symlog_residual=False,
    symlog_linscale=1.0,
    symlog_label_minor=False,
    symlog_sci_cutoff=1e-2,
    legend_ncol_terms=2,
    xlabel=r"$kL_\mathrm{box}/(2\pi)$", ylabel=r"$P_{\rm 1D}$", residual_ylabel=r"$\Delta$",
)

fig, (ax, ax_res), order_sum_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_p1d_total_recovery(
    Pk_total=Pk_total, Pk_total_from_terms=Pk_total_from_terms, Pk_pair=Pk_pair, k=k, delta_v_list=delta_v_list, L_box=L_box, Npix=Npix,
    ordered_pairs=ordered_pairs,
    show_order_sums=True,
    show_order_sum_residuals=True,
    use_symlog_main=True,
    use_symlog_residual=True,
)


In [ ]:

# ===============================================================
# Compute xi1D

stats.compute_xi1d()

xi_total_from_terms = stats.xi_total_from_terms

xi_total_per_sample = stats.xi_total_per_sample
xi_total = stats.xi_total

xi_pair_per_sample = stats.xi_pair_per_sample
xi_pair = stats.xi_pair

lags = stats.lags
lags_norm = lags / L_box

# ===============================================================
# Plots xi1D

fig, axes, axes_res, xi_order_sum_dict, xi_order_sum_plot_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_xi_by_order(
    ordered_pairs=ordered_pairs, xi_pair=xi_pair, lags_norm=lags_norm, delta_v_list=delta_v_list, L_box=L_box,
    tolerance_threshold=1e-6,
    residual_mode="auto_reference",
    xlim=(-0.01, 0.51),
    show_delta_v_diff_lines=True,
    use_symlog_main=False,
    use_symlog_residual=False,
    symlog_linscale=1.0,
    symlog_base=10,
    symlog_label_minor=False,
    symlog_sci_cutoff=1e-2,
    show_order_badge=True,
    legend_ncol_terms=2,
    xlabel=r"$\Delta v / L_\mathrm{box}$", ylabel=r"$\xi_{\rm 1D}$", residual_ylabel=r"$\Delta$",
)

fig, (ax, ax_res), xi_order_sum_dict, xi_order_sum_plot_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_xi_total_recovery(
    xi_total=xi_total, xi_total_from_terms=xi_total_from_terms, xi_pair=xi_pair, lags_norm=lags_norm, delta_v_list=delta_v_list, L_box=L_box, ordered_pairs=ordered_pairs,
    xlim=(-0.01, 0.51),
    show_order_sums=True,
    show_order_sum_residuals=True,
    show_delta_v_diff_lines=True,
    annotate_half_line=True,
    half_line_text=r"$L_\mathrm{box}/2$",
    use_symlog_main=False,
    use_symlog_residual=True,
    main_symlog_linthresh=None,
    symlog_linscale=1.0,
    symlog_base=10,
    symlog_sci_cutoff=1e-2,
    xlabel=r"$\Delta v / L_\mathrm{box}$", ylabel=r"$\xi_{\rm 1D}$", residual_ylabel=r"$\Delta$",
)